## BigStitcher dataset creation — example notebook

Demonstrates all four public API functions:

1. `create_bigstitcher_dataset()` - copy + re-encode arrays into a new zarr store  
2. `create_bigstitcher_dataset_symlinked()` - symlink existing zarr groups (no data copy)  
3. `get_bigstitcher_interest_points()` - convert .csv file to Bigstitcher interestpoints.n5
4. `add_interest_points_to_xml()` - patch an existing `dataset.xml` with BigStitcher interest points

Replace the paths and array references in each section with your real data.

In [6]:
from pathlib import Path
import zarr
import numpy as np
import dask.array as da

from bigstitcher.to_bigstitcher import (
    create_bigstitcher_dataset,
    create_bigstitcher_dataset_symlinked,
    add_interest_points_to_xml,
)

---
### 1. `create_bigstitcher_dataset()` — 3-D tiles

Each tile is a plain `(z, y, x)` array.  
The output zarr groups will be native 3-D (`indicies="[]"`), one `ViewSetup` per tile.

In [ ]:
# ── Replace with your real tile paths or arrays ────────────────────────────
TILE_PATHS_3D = [
    "path_to_tile0/tile0.zarr/s0",
    "path_to_tile1/tile1.zarr/s0",
    # add more tiles as needed
]
OUTPUT_3D = Path("output_dir_path/bigstitcher/")

# Physical voxel size in (x, y, z) order — adjust to match your microscope
VOXEL_SIZE = (0.259, 0.259, 1.0)   # micrometers
# ──────────────────────────────────────────────────────────────────────────

tiles_3d = [zarr.open(p, mode="r") for p in TILE_PATHS_3D]
print("Tile shapes:", [t.shape for t in tiles_3d])

Tile shapes: [(269, 3892, 2048)]


In [ ]:
result_3d = create_bigstitcher_dataset(
    zarr_arrays=tiles_3d,
    voxel_size=VOXEL_SIZE,
    output_folder=OUTPUT_3D,
    voxel_unit="micrometer",
    tile_names=[f"tile_{i}" for i in range(len(tiles_3d))],
    downsampling_factors=[(2, 2, 2), (4, 4, 4), (8, 8, 8)],
    chunk_size=(64, 128, 128),
    compression="zstd",
    compression_level=3,
    n_workers=4,
    threads_per_worker=2,
    memory_limit="8GB",
)

print("Output:", result_3d)
print("XML:", result_3d / "dataset.xml")

---
### 2. `create_bigstitcher_dataset()` — 4-D multi-channel tiles

Each tile has shape `(c, z, y, x)`.  
The output zarr groups are 5-D `(t=1, c, z, y, x)`.  
One `ViewSetup` is created **per channel per tile**, with `indicies="0 {ch}"`.

In [ ]:
# ── Replace with your real multi-channel tile paths ────────────────────────
TILE_PATHS_4D = [
    "/path/to/multichannel_tile0.zarr",
    "/path/to/multichannel_tile1.zarr",
]
OUTPUT_4D = Path("/path/to/output/dataset_4d")
CHANNEL_NAMES = ["DAPI", "GFP"]    # one name per channel
# ──────────────────────────────────────────────────────────────────────────

tiles_4d = [zarr.open(p, mode="r") for p in TILE_PATHS_4D]
print("Tile shapes (c, z, y, x):", [t.shape for t in tiles_4d])

In [ ]:
result_4d = create_bigstitcher_dataset(
    zarr_arrays=tiles_4d,
    voxel_size=VOXEL_SIZE,
    output_folder=OUTPUT_4D,
    voxel_unit="micrometer",
    channel_names=CHANNEL_NAMES,
    downsampling_factors=[(2, 2, 2), (4, 4, 4), (8, 8, 8)],
    n_workers=4,
    threads_per_worker=2,
    memory_limit="8GB",
)

print("Output:", result_4d)

---
### 3. `create_bigstitcher_dataset()` — 5-D tiles with interest points

Each tile has shape `(t, c, z, y, x)`.  
Pass `interest_points_n5` to embed BigStitcher interest points directly at creation time.

In [ ]:
# ── Replace with your real 5-D tile paths ─────────────────────────────────
TILE_PATHS_5D = [
    "/path/to/timeseries_tile0.zarr",  # shape (t, c, z, y, x)
]
OUTPUT_5D = Path("/path/to/output/dataset_5d")
IP_N5_PATH = "/path/to/interestpoints.n5"   # set to None to skip
# ──────────────────────────────────────────────────────────────────────────

tiles_5d = [zarr.open(p, mode="r") for p in TILE_PATHS_5D]
print("Tile shapes (t, c, z, y, x):", [t.shape for t in tiles_5d])

In [ ]:
result_5d = create_bigstitcher_dataset(
    zarr_arrays=tiles_5d,
    voxel_size=VOXEL_SIZE,
    output_folder=OUTPUT_5D,
    voxel_unit="micrometer",
    channel_names=["DAPI", "GFP"],
    downsampling_factors=[(2, 2, 2), (4, 4, 4)],
    n_workers=4,
    threads_per_worker=2,
    memory_limit="8GB",
    interest_points_n5=IP_N5_PATH,   # omit or set to None if not available
)

print("Output:", result_5d)

---
### 4. `create_bigstitcher_dataset_symlinked()` — no data copy

Use this when your tiles are already well-formatted zarr groups and you don't want to copy data.  
Each source is symlinked as `dataset.zarr/tile_N.zarr`.  

- **3-D source** → `indicies="[]"`  
- **5-D source** → `indicies="0 0"` (single ViewSetup pointing at tp=0, ch=0)

In [ ]:
# ── Replace with your existing zarr paths ─────────────────────────────────
SOURCE_ZARR_PATHS = [
    "path_to_symlinked_zarr_dataset/",
]
OUTPUT_SYMLINKED = Path("path_to_bigstitcher_symlinked_dir/")

VOXEL_SIZE = (0.259, 0.259, 1.0)   # micrometers

# ──────────────────────────────────────────────────────────────────────────

# Inspect the source shape so we know it's compatible
for p in SOURCE_ZARR_PATHS:
    grp = zarr.open_group(p, mode="r")
    ms = grp.attrs.get("multiscales")
    if ms:
        base = grp[ms[0]["datasets"][0]["path"]]
        print(f"{p}: shape={base.shape}, ndim={base.ndim}")
    else:
        print(f"{p}: no multiscales metadata found")

In [ ]:
result_sym = create_bigstitcher_dataset_symlinked(
    zarr_paths=SOURCE_ZARR_PATHS,
    voxel_size=VOXEL_SIZE,
    output_folder=OUTPUT_SYMLINKED,
    voxel_unit="micrometer",
    tile_names=[f"tile_{i}" for i in range(len(SOURCE_ZARR_PATHS))],
    channel_names=["channel_0"],
    # interest_points_n5="/path/to/interestpoints.n5",  # optional
)

print("Output:", result_sym)

# Verify symlinks were created
for link in sorted((result_sym / "dataset.zarr").iterdir()):
    print(f"  {link.name} -> {link.resolve()}")

---
## 5. `add_interest_points_to_xml()` — patch an existing XML

Use this after running BigStitcher's interest-point detection to update a `dataset.xml` that was created without interest points.  
The `<ViewInterestPoints>` section is **replaced** (not appended).

In [ ]:
# ── Replace with your real paths ───────────────────────────────────────────
EXISTING_XML   = OUTPUT_3D / "dataset.xml"          # XML created in section 1
INTEREST_POINTS_N5 = OUTPUT_3D / "interestpoints.n5"  # produced by BigStitcher
# ──────────────────────────────────────────────────────────────────────────

add_interest_points_to_xml(
    xml_path=EXISTING_XML,
    interest_points_n5=INTEREST_POINTS_N5,
)

# Verify the XML was updated
from xml.etree import ElementTree as ET
root = ET.parse(EXISTING_XML).getroot()
vip_files = root.findall("ViewInterestPoints/ViewInterestPointsFile")
print(f"Interest point entries in XML: {len(vip_files)}")
for el in vip_files:
    print(f"  tp={el.get('timepoint')}  setup={el.get('setup')}  label={el.get('label')}")

---
### 6. Quick sanity check — inspect the generated XML

Parse and print key sections of any generated `dataset.xml`.

In [ ]:
from xml.etree import ElementTree as ET

XML_TO_CHECK = OUTPUT_3D / "dataset.xml"   # change as needed

root = ET.parse(XML_TO_CHECK).getroot()

print("=== ViewSetups ===")
for vs in root.findall(".//ViewSetup"):
    sid  = vs.findtext("id")
    name = vs.findtext("name")
    size = vs.findtext("size")
    ch   = vs.findtext(".//channel")
    print(f"  id={sid}  name={name}  size={size}  channel={ch}")

print("\n=== Timepoints ===")
print(" ", root.findtext(".//integerpattern"))

print("\n=== zgroups ===")
for zg in root.findall(".//zgroup"):
    print(f"  setup={zg.get('setup')}  tp={zg.get('tp')}  "
          f"path={zg.get('path')}  indicies={zg.get('indicies')}")

print("\n=== Calibration affines ===")
for vr in root.findall(".//ViewRegistration"):
    tp    = vr.get("timepoint")
    setup = vr.get("setup")
    aff   = vr.findtext(".//affine")
    print(f"  tp={tp}  setup={setup}  affine={aff}")

---
### Bonus 
### 7. Create interest points from CSV files

Use `get_bigstitcher_interest_points()` when your interest point coordinates live in CSV files
(e.g. exported from a segmentation tool) rather than a BigStitcher n5 store.

The function writes a BigStitcher-compatible `interestpoints.n5` and then
`add_interest_points_to_xml()` patches the XML — the same two-step workflow regardless
of whether your image data is 3-D, 4-D, or 5-D.

**Expected CSV columns** (defaults, customisable via `x_col`/`y_col`/`z_col`/`id_col`):

| Column | Meaning |
|---|---|
| `COM X (pixels)` | X coordinate in voxels |
| `COM Y (pixels)` | Y coordinate in voxels |
| `COM Z (pixels)` | Z coordinate in voxels |
| `Object ID` | Unique integer ID per point |

In [ ]:
from bigstitcher.bigstitcher_interest_points import get_bigstitcher_interest_points

# ── Replace with your real CSV paths and output folder ─────────────────────
# One CSV per view setup; list index == viewSetupId.
# Use a dict instead of a list if your setup IDs are not 0-based:
#   setups={0: "tile0.csv", 3: "tile3.csv"}
CSV_PATHS = [
    "/path/to/center_of_masses_tile0.csv",
    "/path/to/center_of_masses_tile1.csv",
]
DATASET_XML = OUTPUT_3D / "dataset.xml"          # XML to patch (created in section 1)
IP_N5_OUT   = OUTPUT_3D / "interestpoints.n5"    # will be created here
# ──────────────────────────────────────────────────────────────────────────

# Step 1 — write interestpoints.n5 from CSV files
ip_n5 = get_bigstitcher_interest_points(
    setups=CSV_PATHS,
    output_path=IP_N5_OUT,
    label="beads",          # group name inside each tpId_*/viewSetupId_* group
    timepoint=0,            # timepoint index
    # x_col="COM X (pixels)",   # default column names — change if your CSV differs
    # y_col="COM Y (pixels)",
    # z_col="COM Z (pixels)",
    # id_col="Object ID",
)

# Step 2 — patch dataset.xml with the new interest-point entries
add_interest_points_to_xml(
    xml_path=DATASET_XML,
    interest_points_n5=ip_n5,
)

# Verify
from xml.etree import ElementTree as ET
root_xml = ET.parse(DATASET_XML).getroot()
vip_files = root_xml.findall("ViewInterestPoints/ViewInterestPointsFile")
print(f"Interest point entries patched into XML: {len(vip_files)}")
for el in vip_files:
    print(f"  tp={el.get('timepoint')}  setup={el.get('setup')}  label={el.get('label')}")